In [ ]:
# ===== Query Transformation + Routing (Agentic RAG demo, 1 cell) =====
!pip -q install -U sentence-transformers chromadb

import re
import chromadb
from sentence_transformers import SentenceTransformer

# -----------------------------
# 1) Build a tiny knowledge base (RAG)
# -----------------------------
docs = [
    ("kb1", "Add/Drop deadline is Week 2 Friday. Source: Student Handbook §2.1"),
    ("kb2", "A prerequisite is a module you must complete before taking another module. Source: Academic Rules §3.2"),
    ("kb3", "Contextual compression reduces noise by extracting only relevant sentences from retrieved chunks."),
]

embed = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
col = client.get_or_create_collection("kb_router_demo")

ids = [i for i,_ in docs]
texts = [t for _,t in docs]
embs = embed.encode(texts, normalize_embeddings=True).tolist()

try:
    col.delete(ids=ids)
except Exception:
    pass
col.add(ids=ids, documents=texts, embeddings=embs)

def rag_retrieve(query: str, k: int = 2):
    q_emb = embed.encode([query], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q_emb, n_results=k)
    return list(zip(res["ids"][0], res["documents"][0]))

# -----------------------------
# 2) Query Transformation (rewrite query for better retrieval)
#    (Simple version: keyword-focused rewrite)
# -----------------------------
def transform_query(user_q: str) -> str:
    q = user_q.strip()

    # remove fillers / make it retrieval-friendly
    q = re.sub(r"\b(please|can you|could you|i want to know|tell me)\b", "", q, flags=re.I).strip()

    # add important keywords if detected
    if "add" in q.lower() and "drop" in q.lower():
        q += " deadline academic calendar handbook"
    if "prerequisite" in q.lower():
        q += " definition academic rules module requirements"

    return q

# -----------------------------
# 3) Routing Decision
#    Route to KB if it looks like university handbook topics;
#    otherwise route to external search.
# -----------------------------
KB_TOPICS = ["add/drop", "deadline", "prerequisite", "handbook", "academic rules", "module"]

def route(user_q: str):
    ql = user_q.lower()
    if any(t in ql for t in KB_TOPICS):
        return "rag_kb"
    # simple heuristic for out-of-KB queries
    if any(t in ql for t in ["latest", "today", "news", "price", "2026", "current", "release"]):
        return "web_search"
    return "rag_kb"  # default to KB first

# -----------------------------
# 4) External Search Tool (stub)
#    Replace this with real Google/Tavily/SerpAPI later.
# -----------------------------
def web_search_stub(query: str):
    # Teaching stub: returns pretend results
    return [
        {"title": "Example Web Result 1", "snippet": f"Top result snippet for: {query}"},
        {"title": "Example Web Result 2", "snippet": "Another snippet (replace with real web search)."},
    ]

# -----------------------------
# 5) Agentic RAG Orchestrator
# -----------------------------
def agentic_answer(user_q: str):
    decision = route(user_q)
    tq = transform_query(user_q)

    print("USER QUERY:", user_q)
    print("ROUTE TO :", decision)
    print("TRANSFORM:", tq)

    if decision == "rag_kb":
        hits = rag_retrieve(tq, k=2)
        if not hits:
            return "No KB results found."
        # naive answer: return top hit text
        return "KB Answer:\n" + hits[0][1]

    else:
        results = web_search_stub(tq)
        return "Web Search Results (stub):\n" + "\n".join([f"- {r['title']}: {r['snippet']}" for r in results])

# -----------------------------
# 6) Demo questions
# -----------------------------
qs = [
    "Please tell me what is the add/drop deadline?",
    "What does prerequisite mean?",
    "What is the latest OpenAI model today?",   # should route to web_search
]

for q in qs:
    print("\n" + "="*70)
    print(agentic_answer(q))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


USER QUERY: Please tell me what is the add/drop deadline?
ROUTE TO : rag_kb
TRANSFORM: what is the add/drop deadline? deadline academic calendar handbook
KB Answer:
Add/Drop deadline is Week 2 Friday. Source: Student Handbook §2.1

USER QUERY: What does prerequisite mean?
ROUTE TO : rag_kb
TRANSFORM: What does prerequisite mean? definition academic rules module requirements
KB Answer:
A prerequisite is a module you must complete before taking another module. Source: Academic Rules §3.2

USER QUERY: What is the latest OpenAI model today?
ROUTE TO : web_search
TRANSFORM: What is the latest OpenAI model today?
Web Search Results (stub):
- Example Web Result 1: Top result snippet for: What is the latest OpenAI model today?
- Example Web Result 2: Another snippet (replace with real web search).
